In [ ]:
# --- 1. Setup and Imports ---
import pandas as pd
import numpy as np
import os
import joblib
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Any

# ML Metrics and Calibration
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
    root_mean_squared_error,
    accuracy_score,
    classification_report
)
from sklearn.calibration import calibration_curve, CalibrationDisplay

# --- Add project root to sys.path if needed ---
import sys
try:
    # Assumes notebook is in 'notebooks' directory, adjust if different
    PROJECT_ROOT_PATH = Path('.').resolve().parent
    if str(PROJECT_ROOT_PATH) not in sys.path:
        sys.path.append(str(PROJECT_ROOT_PATH))
    print(f"Project Root added to sys.path: {PROJECT_ROOT_PATH}")
except Exception as e:
     print(f"Could not automatically determine project root: {e}")
     print("Please ensure necessary modules are importable or adjust PROJECT_ROOT_PATH.")
     PROJECT_ROOT_PATH = Path('.') # Fallback

# --- Import shared utilities ---
try:
    from models.utils.features import BaseFeatureConfig, get_feature_config
    # *** IMPORTANT: Ensure this path is correct for your probability calculation helper ***
    from models.utils.probabilities import calculate_poisson_outcome_probs
    print("Helper functions imported.")
except ImportError as e:
    print(f"ERROR importing helper functions: {e}")
    print("Ensure calculate_poisson_outcome_probs is correctly located and importable.")
    # Define a dummy function if import fails, to allow partial notebook execution
    def calculate_poisson_outcome_probs(lambda_home: float, lambda_away: float, max_goals: int = 8) -> Dict:
        warnings.warn("Using dummy calculate_poisson_outcome_probs!")
        return {'prob_H': 0.4, 'prob_D': 0.3, 'prob_A': 0.3, 'prob_O25': 0.5, 'prob_BTTS_Y': 0.5} # Example dummy output

# --- Configuration ---
BASE_DIR = PROJECT_ROOT_PATH
DATA_OUTPUT_DIR = BASE_DIR / 'models' / 'data' / 'outputs'
MODELS_SAVE_DIR = DATA_OUTPUT_DIR / 'joblib'
STACKER_OUTPUT_DIR = DATA_OUTPUT_DIR / 'stacker_outputs'

# --- Select which dataset version to analyze ---
# Change this to True to analyze the 'with_odds' version
ANALYZE_WITH_ODDS = False
odds_suffix = "with_odds" if ANALYZE_WITH_ODDS else "without_odds"

# Input paths
OOF_INPUT_PATH = DATA_OUTPUT_DIR / f'level0_oof_predictions_pca_{odds_suffix}.parquet'
STACKER_MODEL_PATH = MODELS_SAVE_DIR / f'stacker_lgbm_lambda_{odds_suffix}_v1.joblib'

# Level 0 Models included in the OOF file
LEVEL0_MODELS = ["poisson", "random_forest", "gradient_boosting", "monte_carlo"] # <<< Must match OOF generation

# --- Define Target Columns and Key Markets for Analysis ---
# Get feature config to know target column names
feature_cfg = get_feature_config(include_odds=ANALYZE_WITH_ODDS)
TARGET_HG = feature_cfg.target_home_goals # e.g., 'FTHG'
TARGET_AG = feature_cfg.target_away_goals # e.g., 'FTAG'
TARGET_FTR = feature_cfg.target_result   # e.g., 'FTR' (contains 'H', 'D', 'A')

# Key markets to evaluate probabilities for
PROB_MARKETS_TO_EVALUATE = {
    '1X2': ['prob_H', 'prob_D', 'prob_A'],
    'OU15': ['prob_O15', 'prob_U15'],
    'OU25': ['prob_O25', 'prob_U25'],
    'OU35': ['prob_O35', 'prob_U35'],
    'OU45': ['prob_O45', 'prob_U45'],
    'BTTS': ['prob_BTTS_Y', 'prob_BTTS_N'],
    # Add more complex markets if desired, e.g.:
    # 'H_BTTS_N': ['prob_H_and_BTTS_N']
}

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

print(f"--- Configuration Complete - Analyzing '{odds_suffix}' data ---")

In [ ]:
# --- 2. Load Level 0 OOF Data ---
print(f"\n--- Loading Level 0 OOF Data from: {OOF_INPUT_PATH} ---")
assert OOF_INPUT_PATH.exists(), f"OOF data file not found at {OOF_INPUT_PATH}"
oof_df = pd.read_parquet(OOF_INPUT_PATH, engine='pyarrow')
print(f"OOF data loaded. Shape: {oof_df.shape}")
print("Columns:", oof_df.columns.tolist())


In [ ]:
# --- 3. Data Validation and Preparation ---
print("\n--- Validating OOF Data ---")
# Check for necessary columns
required_cols = [TARGET_HG, TARGET_AG, TARGET_FTR]
for model in LEVEL0_MODELS:
    required_cols.extend([f"{model}_expected_HG", f"{model}_expected_AG"])
    for market_probs in PROB_MARKETS_TO_EVALUATE.values():
        for prob_suffix in market_probs:
             required_cols.append(f"{model}_{prob_suffix}")

missing_cols = [col for col in required_cols if col not in oof_df.columns]
assert not missing_cols, f"OOF DataFrame is missing required columns: {missing_cols}"
print("Required columns found.")

# Check for NaNs (should have been handled by OOF script, but double-check)
nan_counts = oof_df[required_cols].isnull().sum()
if nan_counts.sum() > 0:
    warnings.warn(f"NaNs detected in required OOF columns:\n{nan_counts[nan_counts > 0]}")
    # Optional: Add imputation here if needed, though OOF script should handle it
    # oof_df = oof_df.fillna(oof_df.mean(numeric_only=True)) # Example: Mean impute
else:
    print("No NaNs found in required columns.")

# Create Binary Target Columns for Metrics
print("Creating binary target columns for evaluation...")
oof_df['target_H'] = (oof_df[TARGET_FTR] == 'H').astype(int)
oof_df['target_D'] = (oof_df[TARGET_FTR] == 'D').astype(int)
oof_df['target_A'] = (oof_df[TARGET_FTR] == 'A').astype(int)
oof_df['target_O25'] = ((oof_df[TARGET_HG] + oof_df[TARGET_AG]) > 2.5).astype(int)
oof_df['target_U25'] = ((oof_df[TARGET_HG] + oof_df[TARGET_AG]) < 2.5).astype(int)
oof_df['target_BTTS_Y'] = ((oof_df[TARGET_HG] > 0) & (oof_df[TARGET_AG] > 0)).astype(int)
oof_df['target_BTTS_N'] = ((oof_df[TARGET_HG] == 0) | (oof_df[TARGET_AG] == 0)).astype(int)
print("Binary targets created.")

In [ ]:
# --- 4. Level 0 Model Performance Analysis ---
print("\n--- Analyzing Level 0 Model Performance (OOF) ---")
level0_metrics = []

for model_prefix in LEVEL0_MODELS:
    print(f"  Calculating metrics for: {model_prefix}")
    model_results = {'model': model_prefix}

    # Lambda RMSE
    lambda_h_col = f"{model_prefix}_expected_HG"
    lambda_a_col = f"{model_prefix}_expected_AG"
    if lambda_h_col in oof_df.columns and lambda_a_col in oof_df.columns:
        rmse_h = root_mean_squared_error(oof_df[TARGET_HG], oof_df[lambda_h_col])
        rmse_a = root_mean_squared_error(oof_df[TARGET_AG], oof_df[lambda_a_col])
        model_results['RMSE_Lambda_H'] = rmse_h
        model_results['RMSE_Lambda_A'] = rmse_a
        model_results['RMSE_Lambda_Avg'] = (rmse_h + rmse_a) / 2.0
    else:
        warnings.warn(f"Lambda columns not found for {model_prefix}")

    # Probability Metrics (LogLoss, Brier, AUC)
    for market, prob_suffixes in PROB_MARKETS_TO_EVALUATE.items():
        if market == '1X2':
            # Multi-class LogLoss
            prob_cols = [f"{model_prefix}_{p}" for p in prob_suffixes]
            if all(p in oof_df.columns for p in prob_cols):
                # Ensure probabilities sum close to 1 and handle potential clipping issues
                probs_1x2 = oof_df[prob_cols].clip(1e-15, 1 - 1e-15)
                probs_1x2 = probs_1x2.div(probs_1x2.sum(axis=1), axis=0) # Normalize row-wise
                try:
                    model_results['LogLoss_1X2'] = log_loss(oof_df[TARGET_FTR], probs_1x2, labels=['A', 'D', 'H'])
                except ValueError as e:
                    print(f"    WARN: LogLoss calc failed for {model_prefix}_1X2: {e}")
            # AUC/Brier for individual H, D, A
            for prob_suffix, target_suffix in zip(['prob_H', 'prob_D', 'prob_A'], ['target_H', 'target_D', 'target_A']):
                 prob_col = f"{model_prefix}_{prob_suffix}"
                 target_col = target_suffix
                 if prob_col in oof_df.columns:
                     try:
                         model_results[f'AUC_{prob_suffix}'] = roc_auc_score(oof_df[target_col], oof_df[prob_col])
                         model_results[f'Brier_{prob_suffix}'] = brier_score_loss(oof_df[target_col], oof_df[prob_col])
                     except ValueError as e:
                          print(f"    WARN: AUC/Brier calc failed for {prob_col}: {e}")

        else: # Binary markets (OU25, BTTS)
            # Assuming first suffix is 'Yes'/'Over', second is 'No'/'Under'
            prob_y_suffix, prob_n_suffix = prob_suffixes[0], prob_suffixes[1]
            target_y_suffix = f"target_{market}_Y" if 'BTTS' in market else f"target_{market[0]}{market[1:]}" # e.g., target_O25
            target_n_suffix = f"target_{market}_N" if 'BTTS' in market else f"target_{market[0]}{market[1:]}" # e.g., target_U25

            prob_y_col = f"{model_prefix}_{prob_y_suffix}"
            prob_n_col = f"{model_prefix}_{prob_n_suffix}" # Needed for LogLoss check
            target_y_col = target_y_suffix

            if prob_y_col in oof_df.columns and prob_n_col in oof_df.columns:
                 # Check consistency
                 if not np.allclose(oof_df[prob_y_col] + oof_df[prob_n_col], 1.0, atol=1e-3):
                     warnings.warn(f"{model_prefix} {market} probs don't sum to 1. Check calculation.")
                 try:
                     model_results[f'LogLoss_{market}'] = log_loss(oof_df[target_y_col], oof_df[prob_y_col].clip(1e-15, 1 - 1e-15))
                     model_results[f'AUC_{market}'] = roc_auc_score(oof_df[target_y_col], oof_df[prob_y_col])
                     model_results[f'Brier_{market}'] = brier_score_loss(oof_df[target_y_col], oof_df[prob_y_col])
                 except ValueError as e:
                     print(f"    WARN: Metric calc failed for {model_prefix}_{market}: {e}")

    level0_metrics.append(model_results)

level0_metrics_df = pd.DataFrame(level0_metrics).set_index('model')
print("\nLevel 0 OOF Metrics:")
print(level0_metrics_df.round(4))

In [ ]:
# --- 5. Level 0 Calibration Analysis ---
print("\n--- Analyzing Level 0 Model Calibration (OOF) ---")
calibration_markets = {
    'Home Win': ('prob_H', 'target_H'),
    'Over 2.5': ('prob_O25', 'target_O25'),
    'BTTS Yes': ('prob_BTTS_Y', 'target_BTTS_Y')
}

n_models = len(LEVEL0_MODELS)
n_markets = len(calibration_markets)
fig_cal_l0, axes_cal_l0 = plt.subplots(n_markets, n_models, figsize=(5 * n_models, 4 * n_markets), sharex=True, sharey=True)
fig_cal_l0.suptitle('Level 0 Model Calibration (OOF Predictions)', fontsize=16, y=1.02)

for i, model_prefix in enumerate(LEVEL0_MODELS):
    for j, (market_name, (prob_suffix, target_col)) in enumerate(calibration_markets.items()):
        ax = axes_cal_l0[j, i]
        prob_col = f"{model_prefix}_{prob_suffix}"

        if prob_col in oof_df.columns:
            y_true = oof_df[target_col]
            y_prob = oof_df[prob_col]
            # Handle potential NaNs in this specific column if imputation wasn't perfect
            valid_idx = y_prob.notna() & y_true.notna()
            if valid_idx.sum() > 0:
                 disp = CalibrationDisplay.from_predictions(y_true[valid_idx], y_prob[valid_idx], n_bins=10, strategy='uniform', ax=ax)
                 ax.set_title(f"{model_prefix}\n{market_name}")
            else:
                 ax.set_title(f"{model_prefix}\n{market_name}\n(No valid data)")
        else:
            ax.set_title(f"{model_prefix}\n{market_name}\n(Preds Missing)")
        ax.plot([0, 1], [0, 1], 'k:', label='Perfectly calibrated')
        ax.legend()

plt.tight_layout(rect=[0, 0, 1, 1]) # Adjust layout
plt.show()


In [ ]:
# --- 6. Load Stacker Model and Generate Predictions ---
print(f"\n--- Loading Stacker Model: {STACKER_MODEL_PATH} ---")
assert STACKER_MODEL_PATH.exists(), f"Stacker model artifact not found at {STACKER_MODEL_PATH}"
stacker_artifact = joblib.load(STACKER_MODEL_PATH)

# Extract components
model_hg_stacker = stacker_artifact['model_hg']
model_ag_stacker = stacker_artifact['model_ag']
stacker_feature_cols = stacker_artifact['feature_columns'] # Features stacker was trained on

print("Stacker models loaded.")
print(f"Stacker expects {len(stacker_feature_cols)} features.")

# Verify stacker features exist in OOF data
missing_stacker_features = [col for col in stacker_feature_cols if col not in oof_df.columns]
assert not missing_stacker_features, f"OOF data is missing features the stacker requires: {missing_stacker_features}"

# Prepare input for stacker prediction
X_stack_oof = oof_df[stacker_feature_cols]

print("Generating Stacker Lambda predictions on OOF data...")
oof_df['stacker_lambda_HG'] = model_hg_stacker.predict(X_stack_oof)
oof_df['stacker_lambda_AG'] = model_ag_stacker.predict(X_stack_oof)

# Clip stacker lambdas
MAX_LAMBDA = 15.0 # Use same clipping as in training/OOF gen
oof_df['stacker_lambda_HG'] = np.clip(oof_df['stacker_lambda_HG'], 1e-9, MAX_LAMBDA)
oof_df['stacker_lambda_AG'] = np.clip(oof_df['stacker_lambda_AG'], 1e-9, MAX_LAMBDA)
print("Stacker Lambda predictions generated and clipped.")

In [ ]:
# --- 7. Derive Probabilities from Stacker Lambdas ---
print("Deriving outcome probabilities from Stacker Lambdas...")
stacker_probs_list = []
# Use apply row-wise (acceptable for inference/analysis on moderate data size)
# Ensure calculate_poisson_outcome_probs is available
for _, row in oof_df[['stacker_lambda_HG', 'stacker_lambda_AG']].iterrows():
    try:
        probs = calculate_poisson_outcome_probs(row['stacker_lambda_HG'], row['stacker_lambda_AG'])
        stacker_probs_list.append(probs)
    except Exception as e:
        # Handle potential errors in probability calculation (e.g., from dummy function)
        print(f"Error calculating stacker probs for row: {e}")
        stacker_probs_list.append({}) # Append empty dict on error

stacker_probs_df = pd.DataFrame(stacker_probs_list, index=oof_df.index)

# Add stacker probabilities to the main OOF DataFrame with prefix
for col in stacker_probs_df.columns:
    oof_df[f'stacker_{col}'] = stacker_probs_df[col]

print(f"Stacker probabilities derived and added to DataFrame.")
# Verify some stacker probability columns were added
stacker_prob_cols_added = [c for c in oof_df.columns if c.startswith('stacker_prob_')]
assert stacker_prob_cols_added, "No stacker probability columns were added!"
print(f"Added {len(stacker_prob_cols_added)} stacker probability columns.")

In [ ]:
# --- 8. Level 1 (Stacker) Performance Analysis ---
print("\n--- Analyzing Level 1 (Stacker) Performance (OOF) ---")
stacker_metrics = {'model': 'Stacker_LGBM'}

# Lambda RMSE
stacker_metrics['RMSE_Lambda_H'] = root_mean_squared_error(oof_df[TARGET_HG], oof_df['stacker_lambda_HG'])
stacker_metrics['RMSE_Lambda_A'] = root_mean_squared_error(oof_df[TARGET_AG], oof_df['stacker_lambda_AG'])
stacker_metrics['RMSE_Lambda_Avg'] = (stacker_metrics['RMSE_Lambda_H'] + stacker_metrics['RMSE_Lambda_A']) / 2.0

# Probability Metrics (LogLoss, Brier, AUC)
for market, prob_suffixes in PROB_MARKETS_TO_EVALUATE.items():
    if market == '1X2':
        prob_cols = [f"stacker_{p}" for p in prob_suffixes]
        if all(p in oof_df.columns for p in prob_cols):
            probs_1x2 = oof_df[prob_cols].clip(1e-15, 1 - 1e-15)
            probs_1x2 = probs_1x2.div(probs_1x2.sum(axis=1), axis=0)
            try:
                stacker_metrics['LogLoss_1X2'] = log_loss(oof_df[TARGET_FTR], probs_1x2, labels=['A', 'D', 'H'])
            except ValueError as e: print(f"    WARN: Stacker LogLoss calc failed for 1X2: {e}")
        for prob_suffix, target_suffix in zip(['prob_H', 'prob_D', 'prob_A'], ['target_H', 'target_D', 'target_A']):
             prob_col = f"stacker_{prob_suffix}"
             target_col = target_suffix
             if prob_col in oof_df.columns:
                 try:
                     stacker_metrics[f'AUC_{prob_suffix}'] = roc_auc_score(oof_df[target_col], oof_df[prob_col])
                     stacker_metrics[f'Brier_{prob_suffix}'] = brier_score_loss(oof_df[target_col], oof_df[prob_col])
                 except ValueError as e: print(f"    WARN: Stacker AUC/Brier calc failed for {prob_col}: {e}")

    else: # Binary markets
        prob_y_suffix, prob_n_suffix = prob_suffixes[0], prob_suffixes[1]
        target_y_suffix = f"target_{market}_Y" if 'BTTS' in market else f"target_{market[0]}{market[1:]}"
        prob_y_col = f"stacker_{prob_y_suffix}"
        prob_n_col = f"stacker_{prob_n_suffix}"
        target_y_col = target_y_suffix

        if prob_y_col in oof_df.columns and prob_n_col in oof_df.columns:
             if not np.allclose(oof_df[prob_y_col] + oof_df[prob_n_col], 1.0, atol=1e-3):
                 warnings.warn(f"Stacker {market} probs don't sum to 1. Check calculation.")
             try:
                 stacker_metrics[f'LogLoss_{market}'] = log_loss(oof_df[target_y_col], oof_df[prob_y_col].clip(1e-15, 1 - 1e-15))
                 stacker_metrics[f'AUC_{market}'] = roc_auc_score(oof_df[target_y_col], oof_df[prob_y_col])
                 stacker_metrics[f'Brier_{market}'] = brier_score_loss(oof_df[target_y_col], oof_df[prob_y_col])
             except ValueError as e: print(f"    WARN: Stacker Metric calc failed for {market}: {e}")

stacker_metrics_df = pd.DataFrame([stacker_metrics]).set_index('model')
print("\nStacker OOF Metrics:")
print(stacker_metrics_df.round(4))

In [ ]:
# --- 9. Stacker Calibration Analysis ---
print("\n--- Analyzing Level 1 (Stacker) Calibration (OOF) ---")

fig_cal_l1, axes_cal_l1 = plt.subplots(1, n_markets, figsize=(5 * n_markets, 4), sharey=True)
if n_markets == 1: axes_cal_l1 = [axes_cal_l1] # Ensure axes is iterable
fig_cal_l1.suptitle('Level 1 Stacker Calibration (OOF Predictions)', fontsize=16, y=1.02)

for j, (market_name, (prob_suffix, target_col)) in enumerate(calibration_markets.items()):
    ax = axes_cal_l1[j]
    prob_col = f"stacker_{prob_suffix}" # Use stacker prefix

    if prob_col in oof_df.columns:
        y_true = oof_df[target_col]
        y_prob = oof_df[prob_col]
        valid_idx = y_prob.notna() & y_true.notna()
        if valid_idx.sum() > 0:
             disp = CalibrationDisplay.from_predictions(y_true[valid_idx], y_prob[valid_idx], n_bins=10, strategy='uniform', ax=ax, name="Stacker")
             ax.set_title(f"Stacker: {market_name}")
        else:
             ax.set_title(f"Stacker: {market_name}\n(No valid data)")
    else:
        ax.set_title(f"Stacker: {market_name}\n(Preds Missing)")
    ax.plot([0, 1], [0, 1], 'k:', label='Perfectly calibrated')
    ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()


In [ ]:
# --- 10. Comparison and Conclusion ---
print("\n--- Comparison: Level 0 vs Level 1 Stacker (OOF Metrics) ---")

# Combine metrics DataFrames
all_metrics_df = pd.concat([level0_metrics_df, stacker_metrics_df])

# Select key metrics for comparison display
metrics_to_compare = [
    'RMSE_Lambda_Avg', 'LogLoss_1X2', 'LogLoss_OU25', 'LogLoss_BTTS',
    'Brier_prob_H', 'Brier_prob_D', 'Brier_prob_A', 'Brier_OU25', 'Brier_BTTS',
    'AUC_prob_H', 'AUC_prob_D', 'AUC_prob_A', 'AUC_OU25', 'AUC_BTTS'
]
# Filter out columns that might not exist if calculations failed
metrics_to_compare = [m for m in metrics_to_compare if m in all_metrics_df.columns]

print(all_metrics_df[metrics_to_compare].round(4))

# --- Interpretation Guidance ---
print("\n--- Interpretation ---")
print("Lower is better for: RMSE, LogLoss, Brier Score.")
print("Higher is better for: AUC.")
print("\nKey things to look for:")
print("- Does the 'Stacker_LGBM' row show consistently better (lower RMSE/LogLoss/Brier, higher AUC) scores than the individual Level 0 models?")
print("- How much improvement does the stacker provide over the *best* Level 0 model for each metric?")
print("- Check the calibration plots: Is the stacker's calibration curve closer to the diagonal (perfect calibration) than the Level 0 models?")
print("- Review feature importance (from train_stacker.py output CSV): Which Level 0 models/predictions did the stacker rely on most?")

